# 074 — Objetivos de preentrenamiento

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución de referencia

**Ejercicio 1.** L = −(ln 0,5 + ln 0,25 + ln 0,125) = 0,693 + 1,386 + 2,079 = 4,159.
L media = 4,159/3 ≈ 1,386. PPL = e^1,386 ≈ **4,0**: como dudar uniformemente entre
4 opciones por token.

**Ejercicio 2.** Entrada: `[los, [MASK], aprenden, de, los, [MASK]]`; objetivo:
posición 2 → `modelos`, posición 6 → `datos` (la pérdida solo se computa ahí).
La sustitución aleatoria del 10 % evita que el modelo aprenda a activarse solo
cuando ve `[MASK]`, símbolo que nunca aparece en inferencia.

**Ejercicio 3.** Entrada: `el entrenamiento <X> requiere <Y>`.
Objetivo: `<X> distribuido <Y> sincronizar gradientes <Z>`.

**Ejercicio 4.** El contrato (`kind`, `evidence`, `limitations`) es estable; los
valores derivados del muestreo con semilla cambiarían con otra semilla.

In [ ]:
import math

# Ejercicio 1
p = [0.5, 0.25, 0.125]
L_total = -sum(math.log(x) for x in p)
L_media = L_total / len(p)
ppl = math.exp(L_media)
print(f"L_total={L_total:.3f} L_media={L_media:.3f} PPL={ppl:.2f}")  # 4.159, 1.386, 4.00

# Ejercicio 2
entrada_mlm = ["los", "[MASK]", "aprenden", "de", "los", "[MASK]"]
objetivo_mlm = {2: "modelos", 6: "datos"}

# Ejercicio 3
entrada_t5 = "el entrenamiento <X> requiere <Y>"
objetivo_t5 = "<X> distribuido <Y> sincronizar gradientes <Z>"

# Ejercicio 4
result = run_lab("llm", seed=74)
assert result["kind"] == "llm"
assert result["evidence"] and result["limitations"]
show(result)

## Reflexión

1. ¿Por qué el objetivo causal aprovecha el 100 % de las posiciones como señal y MLM
   solo ~15 %, y qué implica eso para el costo de alcanzar una misma calidad?
2. Un modelo preentrenado solo con next-token completa "La capital de Francia es"
   correctamente pero responde mal a "¿Cuál es la capital de Francia?". ¿Qué explica
   la diferencia y qué etapa posterior la corrige?
3. Si tu tarea es clasificar 50 000 tickets de soporte con presupuesto mínimo,
   ¿qué familia (GPT, BERT, T5) elegirías y por qué?